# 生存分析模型比较实验 - 两侧CSA
## 36组数据 × 4种方法（Two-sided Conformalized Survival Analysis）

In [ ]:
import numpy as np
import pandas as pd
np.random.seed(2026)

import warnings
warnings.filterwarnings('ignore')
import importlib
import config
importlib.reload(config)
from config import *
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = ['Heiti TC']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 生成36组数据
random_seed = 2026
sample_sizes = [100, 500, 2000]
cens_lambdas = [0.1, 0.3, 0.5]
p_list = [1, 50]
hetero_list = [False, True]

datasets = []
configs = []

for n in sample_sizes:
    for cens in cens_lambdas:
        for p in p_list:
            for hetero in hetero_list:
                X, surv, time, event, real_censor = generate_weibull_data(
                    n=n, p=p, hetero=hetero, cens_rate=cens
                )
                datasets.append((X, surv, time, event))
                configs.append({
                    'n': n, 'cens_rate': cens, 'p': p, 
                    'hetero': hetero, 'real_censor': real_censor
                })

print(f'✅ 生成完成：{len(datasets)}组数据')

In [ ]:
# 批量拟合四种模型
results = []

for i, (X, surv, time, event) in enumerate(datasets):
    cfg = configs[i]
    print(f"第 {i+1} 组: n={cfg['n']}, p={cfg['p']}, cens={cfg['cens_rate']}, hetero={cfg['hetero']}")

    # 1. Kaplan-Meier
    kmf = fit_kaplan_meier(time, event)
    
    # 2. Cox
    cox = fit_cox(X, time, event)
    c_cox = evaluate_model(cox, X, time, event, 'cox')
    
    # 3. Weibull
    weibull = fit_weibull(X, time, event)
    c_weibull = evaluate_model(weibull, X, time, event, 'weibull')
    
    # 4. 随机生存森林
    rsf = fit_rsf(X, time, event)
    c_rsf = evaluate_model(rsf, X, time, event, 'rsf')
    
    results.append({
        **cfg,
        'C_Cox': c_cox,
        'C_Weibull': c_weibull,
        'C_RSF': c_rsf
    })
print("数据处理完成")
df_results = pd.DataFrame(results)
df_results

In [ ]:
# 保存结果
df_results.to_csv('模型拟合结果.csv', index=False)
print('✅ 结果已保存')

In [ ]:
# 两侧CSA实验：混合区间方法（修复版本）
# 现在使用conformal p-value进行分类（而非简单阈值）
# 这保证了 P(Type I error) = α/2 的有限样本覆盖
# - 未删失样本(Δ=1)：两侧区间[L, U]，覆盖率 α/2
# - 删失样本(Δ=0)：单侧区间[L, ∞)，覆盖率 α/2

csa_results = []

# 基础模型选择：KM、Cox、Weibull、RSF
base_models = ['km', 'cox', 'weibull', 'rsf']
alpha = 0.1

for i, (X, surv, time, event) in enumerate(datasets):
    cfg = configs[i]

    print(f"CSA 第 {i+1} 组: n={cfg['n']}, p={cfg['p']}, cens={cfg['cens_rate']}, hetero={cfg['hetero']}")
    X_train, time_train, event_train, X_cal, time_cal, event_cal, X_test, time_test, event_test = split_survival_data(
        X, time, event, test_size=0.2, cal_size=0.25, random_state=2026
    )

    kmf = fit_kaplan_meier(time_train, event_train)
    cox = fit_cox(X_train, time_train, event_train)
    weibull = fit_weibull(X_train, time_train, event_train)
    rsf = fit_rsf(X_train, time_train, event_train)

    model_map = {
        'km': kmf,
        'cox': cox,
        'weibull': weibull,
        'rsf': rsf
    }

    for model_type in base_models:
        model = model_map[model_type]
        
        # 使用两侧CSA方法：混合区间（关键修复：现在传入训练集用于分类器）
        lower, upper, q_value, classification = fit_csa_intervals_two_sided(
            model, X_train, time_train, event_train,
            X_cal, time_cal, event_cal, X_test,
            alpha=alpha, model_type=model_type
        )
        
        # 使用两侧CSA的覆盖率评估函数
        metrics = evaluate_interval_coverage_two_sided(lower, upper, time_test, event_test, classification)

        csa_results.append({
            'n': cfg['n'],
            'p': cfg['p'],
            'cens_rate': cfg['cens_rate'],
            'hetero': cfg['hetero'],
            'model_type': model_type,
            'method': 'Two-sided CSA (Conformal p-value)',
            'alpha': alpha,
            'coverage': metrics['coverage'],
            'coverage_two_sided': metrics['coverage_two_sided'],
            'coverage_one_sided': metrics['coverage_one_sided'],
            'mean_width_two_sided': metrics['mean_width_two_sided'],
            'q_value': q_value,
            'num_two_sided': metrics['num_two_sided'],
            'num_one_sided': metrics['num_one_sided']
        })


# 保存两侧CSA实验结果
df_csa = pd.DataFrame(csa_results)
df_csa.to_csv('Two_sided_CSA_results_fixed.csv', index=False)
print('两侧CSA实验完成（修复版本）：', df_csa.shape[0], '条记录')

# 简要汇总
print('\n两侧CSA汇总：按模型类型查看平均指标')
print(df_csa.groupby('model_type')[['coverage','coverage_two_sided','coverage_one_sided','mean_width_two_sided']].mean())
print('\n无限上界比例（样本分布）：')
print(df_csa.groupby('model_type')[['num_two_sided','num_one_sided']].mean())

## 两侧CSA方法说明

### 方法原理

两侧CSA采用分类策略，区分未删失和删失样本：

1. **删失状态分类**：使用随机森林分类器预测P(Δ=1|X)，识别哪些样本的生存时间被完全观测

2. **混合区间生成**：
   - 预测为**未删失(Δ=1)**的样本：构造两侧区间 [L, U]
   - 预测为**删失(Δ=0)**的样本：构造单侧区间 [L, ∞)
   
3. **覆盖率保证**（有限样本）：
   - 对两侧区间分配 α/2
   - 对单侧区间分配 α/2
   - 整体：P(T ∉ Ĉ(X)) ≤ α

### 关键特性

| 特性 | 描述 |
|------|------|
| **上界计算** | 通过反演生存函数 F̂⁻¹(0.5 + q) 得到，超出支撑时为∞ |
| **避免权重** | 无需估计删失机制 ĉ(x)，规避数值不稳定性 |
| **有限样本** | 基于conformal p-value的分类，提供有限样本保证 |
| **混合覆盖** | 利用未删失样本的完整信息提供更精确的两侧区间 |

### 与传统CSA的比较

| 维度 | 传统CSA | 两侧CSA |
|------|----------|----------|
| **区间形式** | 所有[L, ∞) | 混合：两侧 + 单侧 |
| **权重估计** | 需要 | 不需要 |
| **上界** | 无 | 有限 |
| **覆盖类型** | 统一 | 分群体 |
| **理论样本量** | 渐近 | **有限样本** |
| **数值稳定性** | 中 | 好 |

### 实验输出

- **coverage**: 总体覆盖率
- **coverage_two_sided**: 两侧区间的覆盖率（应≥ α/2）
- **coverage_one_sided**: 单侧区间的覆盖率（应≥ α/2）
- **mean_width_two_sided**: 两侧区间的平均宽度
- **num_two_sided**: 获得两侧区间的样本数
- **num_one_sided**: 获得单侧区间的样本数